# Milestone-2 RAG Architecture + Application

This notebook calls the modular application code. Configure `.env`, upload supported documents to S3, and run the cells in order.

In [1]:
# !pip install pymupdf
# Install this dependencies to run the PDFs from Bucket as We're not using Textract service.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from dotenv import load_dotenv
from rag_app.bootstrap import build_services
from rag_app.config import Settings

load_dotenv(project_root / '.env')
settings = Settings.from_env()
services = build_services(settings)
print(f'Using index: {settings.opensearch_index}')

Using index: rag-vector-search


## Ingest S3 documents
PymuPDF performs asynchronous text extraction, LangChain splits the extracted lines, Bedrock creates vectors, and OpenSearch indexes the vectors.

In [2]:
report = services.ingestion.ingest(settings.s3_bucket, settings.s3_prefix)
report

IngestionReport(documents_seen=5, documents_ingested=5, chunks_indexed=330, errors=[])

## Ask a grounded question
The answer is generated only from retrieved OpenSearch chunks and includes source labels.

In [3]:
result = services.rag.ask('What is attention mechanism', top_k=settings.retrieval_top_k)
print(result.answer)
[(f's3://{item.source_bucket}/{item.source_key}', item.score) for item in result.sources]

An attention mechanism can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with the corresponding key [1][2][3][4].


[('s3://rag-milestone-bucket/documents/doc_1_attention.pdf', 0.7242729),
 ('s3://rag-milestone-bucket/documents/doc_1_attention.pdf', 0.7242729),
 ('s3://rag-milestone-bucket/documents/doc_1_attention.pdf', 0.7216673),
 ('s3://rag-milestone-bucket/documents/doc_1_attention.pdf', 0.7216673)]

In [4]:
result = services.rag.ask('What is BERT ? Explain Encoder and Decoder in short.', top_k=settings.retrieval_top_k)
print(result.answer)
[(f's3://{item.source_bucket}/{item.source_key}', item.score) for item in result.sources]

BERT (Bidirectional Encoder Representations from Transformers) is a linguistic model presented by a group of scientists from the Google AI Language laboratory under the leadership of J. Devlin at the end of 2018 [3][4]. It is intended for deep preliminary learning of bidirectional text representation for subsequent use in machine learning models [3][4].

The BERT architecture is based on the multilayer bidirectional transformer described in 2017 by A. Washwani [1][2]. It uses text embeddings described in 2016 to represent an input sequence [1][2]. 

However, the context does not provide specific details about the Encoder and Decoder components of BERT. Therefore, I do not know the specific roles or functionalities of the Encoder and Decoder in BERT based on the provided context [1][2][3][4].


[('s3://rag-milestone-bucket/documents/doc_4_bert.pdf', 0.7804843),
 ('s3://rag-milestone-bucket/documents/doc_4_bert.pdf', 0.78048426),
 ('s3://rag-milestone-bucket/documents/doc_4_bert.pdf', 0.75737953),
 ('s3://rag-milestone-bucket/documents/doc_4_bert.pdf', 0.75737953)]

In [5]:
result = services.rag.ask('What RAG? Explain Ingestion pipeline in short', top_k=settings.retrieval_top_k)
print(result.answer)
[(f's3://{item.source_bucket}/{item.source_key}', item.score) for item in result.sources]

The context provided does not contain a detailed explanation of the RAG ingestion pipeline. However, it does mention the fundamental framework of RAG, which involves combining the generative capabilities of LLMs (Large Language Models) with external knowledge retrieved from a separate database, such as an organizational database [3][4]. 

The RAG pipeline begins with a query and ends with a result, with three fundamental parts in between: retrieval, augmentation, and generation. The retrieval part involves searching for the most relevant information, the augmentation part is the output of the retriever and serves as the input for the generator, and the generation part produces the final result [1][2][3][4].

The embedding model is used to translate data of different modalities into a vector, and the same embedding model must be used for both the vector database and the input query to measure similarity [1][2][3][4]. 

However, the specific details of the ingestion pipeline, such as how

[('s3://rag-milestone-bucket/documents/doc_5_rag.pdf', 0.805408),
 ('s3://rag-milestone-bucket/documents/doc_5_rag.pdf', 0.805408),
 ('s3://rag-milestone-bucket/documents/doc_5_rag.pdf', 0.7199539),
 ('s3://rag-milestone-bucket/documents/doc_5_rag.pdf', 0.7199539)]

In [6]:
result = services.rag.ask('What is CI CD?', top_k=settings.retrieval_top_k)
print(result.answer)
[(f's3://{item.source_bucket}/{item.source_key}', item.score) for item in result.sources]

Continuous Integration/Continuous Deployment (CI/CD) refers to practices that are essential for CI/CD pipelines, offering consistent and reliable environments from development to production. This facilitates continuous integration and deployment [1][2].


[('s3://rag-milestone-bucket/documents/doc_3_aws.pdf', 0.629109),
 ('s3://rag-milestone-bucket/documents/doc_3_aws.pdf', 0.629109),
 ('s3://rag-milestone-bucket/documents/doc_4_bert.pdf', 0.583619),
 ('s3://rag-milestone-bucket/documents/doc_4_bert.pdf', 0.583619)]

In [7]:
result = services.rag.ask('What is Langchain ? Why and How it is used in Generative AI', top_k=settings.retrieval_top_k)
print(result.answer)
[(f's3://{item.source_bucket}/{item.source_key}', item.score) for item in result.sources]

LangChain is a rapidly emerging framework that offers a versatile and modular approach to developing applications powered by large language models (LLMs) [3]. By leveraging LangChain, developers can simplify complex stages of the application lifecycle—such as development, productionization, and deployment—making it easier to build scalable, stateful, and contextually aware applications [3]. 

LangChain provides tools for handling chat models, integrating retrieval-augmented generation (RAG), and offering secure API interactions [3]. With LangChain, rapid deployment of sophisticated LLM solutions across diverse domains becomes feasible [3]. 

LangChain is used in Generative AI to streamline the development process, enhance usability, and ensure security in building applications that leverage LLMs [1][3]. It allows developers to focus on creating innovative and secure applications by managing the complexities involved in integrating and deploying LLMs [1][3].


[('s3://rag-milestone-bucket/documents/doc_2_langchain.pdf', 0.86906445),
 ('s3://rag-milestone-bucket/documents/doc_2_langchain.pdf', 0.86906445),
 ('s3://rag-milestone-bucket/documents/doc_2_langchain.pdf', 0.84037906),
 ('s3://rag-milestone-bucket/documents/doc_2_langchain.pdf', 0.84037906)]